In [ ]:
!pip install nnunetv2 SimpleITK tqdm

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [ ]:
from pathlib import Path
import os, json, shutil
import SimpleITK as sitk
import numpy as np
from tqdm import tqdm

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/nnU-NET-pancreas")

RAW = BASE / "nnUNet_raw"
PRE = BASE / "nnUNet_preprocessed"
RES = BASE / "nnUNet_results"

for p in [RAW, PRE, RES]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(RAW)
os.environ["nnUNet_preprocessed"] = str(PRE)
os.environ["nnUNet_results"] = str(RES)

DATASET_ID = 902
DATASET_NAME = f"Dataset{DATASET_ID}_PancreasROI"

DATASET = RAW / DATASET_NAME

IMAGES_TR = DATASET / "imagesTr"
LABELS_TR = DATASET / "labelsTr"

IMAGES_TR.mkdir(parents=True, exist_ok=True)
LABELS_TR.mkdir(parents=True, exist_ok=True)

In [ ]:
images = sorted(SRC_IMAGES.glob("*.mha"))
labels = sorted(SRC_LABELS.glob("*.mha"))

print("Images:", len(images))
print("Labels:", len(labels))

for img_path, lbl_path in tqdm(zip(images, labels), total=len(images)):
    case_id = img_path.stem.replace("_0000", "")

    out_img = IMAGES_TR / f"{case_id}_0000.mha"
    shutil.copy(str(img_path), str(out_img))

    lbl_img = sitk.ReadImage(str(lbl_path))
    lbl_arr = sitk.GetArrayFromImage(lbl_img)

    pancreas_roi = (lbl_arr > 0).astype(np.uint8)

    out_lbl_img = sitk.GetImageFromArray(pancreas_roi)
    out_lbl_img.CopyInformation(lbl_img)

    sitk.WriteImage(out_lbl_img, str(LABELS_TR / f"{case_id}.mha"))

print("Done")
print("imagesTr:", len(list(IMAGES_TR.glob('*.mha'))))
print("labelsTr:", len(list(LABELS_TR.glob('*.mha'))))

Images: 92
Labels: 92


100%|██████████| 92/92 [00:12<00:00,  7.30it/s]

Done
imagesTr: 92
labelsTr: 92


In [ ]:
dataset_json = {
    "channel_names": {
        "0": "MR"
    },
    "labels": {
        "background": 0,
        "pancreas_roi": 1
    },
    "numTraining": len(list(LABELS_TR.glob("*.mha"))),
    "file_ending": ".mha",
    "name": DATASET_NAME,
    "description": "Binary whole pancreas ROI segmentation: tumor + pancreas"
}

with open(DATASET / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=4)

print(json.dumps(dataset_json, indent=4))

{
    "channel_names": {
        "0": "MR"
    },
    "labels": {
        "background": 0,
        "pancreas_roi": 1
    },
    "numTraining": 92,
    "file_ending": ".mha",
    "name": "Dataset902_PancreasROI",
    "description": "Binary whole pancreas ROI segmentation: tumor + pancreas"
}


In [ ]:
for p in sorted(LABELS_TR.glob("*.mha"))[:5]:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(p)))
    print(p.name, np.unique(arr), int(arr.sum()))

10000_0001.mha [0 1] 18615
10001_0001.mha [0 1] 22998
10002_0001.mha [0 1] 12550
10006_0001.mha [0 1] 23313
10007_0001.mha [0 1] 38101


In [ ]:
!nnUNetv2_plan_and_preprocess -d 902 --verify_dataset_integrity

Fingerprint extraction...
Dataset902_PancreasROI
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [ 72. 258. 318.], 3d_lowres: [72, 258, 318]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 32, 'patch_size': (np.int64(320), np.int64(320)), 'median_image_size_in_voxels': array([258., 318.]), 'spacing': array([1.1875, 1.1875]), 'normali

In [ ]:
!nnUNetv2_train 902 3d_fullres 0


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-09 10:02:40.910950: Using torch.compile...
2026-06-09 10:02:41.892186: do_dummy_2d_data_aug: True
2026-06-09 10:02:42.085623: Using splits from existing split file: /content/drive/MyDrive/nnU-NET-pancreas/nnUNet_preprocessed/Dataset902_PancreasROI/splits_final.json
2026-06-09 10:02:42.09

In [ ]:
for p in RES.rglob("checkpoint_best.pth"):

    print(p)

/content/drive/MyDrive/nnU-NET-pancreas/nnUNet_results/Dataset902_PancreasROI/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/checkpoint_best.pth
